In [1]:
import os
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, random_split
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image
import random

In [2]:
class AugmentedECGDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels  # DataFrame containing labels and corresponding IDs
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Extract the image path and ID
        image_path = self.image_paths[idx]
        image_id = os.path.basename(image_path).split('_')[1].split('.')[0]

        # Find the corresponding row in the labels DataFrame using the ID
        label_row = self.labels[self.labels['ID'] == int(image_id)]

        if label_row.empty:
            raise ValueError(f"ID {image_id} not found in the labels DataFrame.")

        # Extract the label values
        labels = label_row.iloc[0, 1:].values

        # Try to load the image
        try:
            image = Image.open(image_path).convert("L")
        except (IOError, OSError) as e:
            print(f"Error loading image {image_path}: {e}")
            return None

        # Convert image to numpy array
        image = np.array(image)

        # Apply transformations
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return {
            "pixel_values": torch.tensor(image, dtype=torch.float32).unsqueeze(0), 
            "labels": torch.tensor(labels, dtype=torch.float32)
        }

In [3]:
# Custom Resize with Anti-Aliasing
class ResizeWithAntiAliasing(A.ImageOnlyTransform):
    def __init__(self, width, height, always_apply=False, p=1.0):
        super(ResizeWithAntiAliasing, self).__init__(always_apply, p)
        self.width = width
        self.height = height

    def apply(self, image, **params):
        # Convert the image to a PIL image, apply resizing with anti-aliasing
        image_pil = Image.fromarray(image)
        image_resized = image_pil.resize((self.width, self.height), Image.Resampling.LANCZOS)
        resized_image_array = np.array(image_resized)
        print(f"Resized image size: {resized_image_array.shape}")

        return np.array(resized_image_array)

In [4]:
train_transform = A.Compose([
    # ResizeWithAntiAliasing(width=224, height=224),  # Resize to 224x224
    A.Resize(height=512, width=512),
    # A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),  # Apply shift, scale, rotate
    # A.GaussianBlur(blur_limit=(3, 7), p=1.0),  # Apply Gaussian blur
    A.Normalize(mean=(0.5,), std=(0.5,)),  # Normalize for single channel
    ToTensorV2(),  # Convert to PyTorch tensor
])

/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/albumentations/core/validation.py:45: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
# Define transformations for validation (no augmentation)
val_transform = A.Compose([
    A.Resize(height=512, width=512),  # Resize to 224x224
    # ResizeWithAntiAliasing(width=224, height=224),  # Resize to 224x224
    A.Normalize(mean=(0.5,), std=(0.5,)),  # Normalize for single channel
    ToTensorV2(),
])

In [6]:
path_to_data = './data/train/'

# Load the CSV file with labels
labels_df = pd.read_csv("train_final.csv")
labels_df = labels_df[labels_df['ID'] < 1000]

# Create a list of image paths based on the 'id' column
image_paths = [path_to_data + f"train_{id_:06d}.png" for id_ in labels_df['ID']]
image_paths = image_paths[:1000]

In [7]:
# remove image 142
image_paths.remove('./data/train/train_000142.png')

In [8]:
# Shuffle the list to randomize the order
random.shuffle(image_paths)

# Calculate the split index (95% for a, 5% for b)
split_index = int(0.95 * len(image_paths))

# Split the list into 'a' and 'b'
valid_image_paths = image_paths[:split_index]  # First 95% for list a
valid_test_images = image_paths[split_index:]  # Remaining 5% for list b

In [9]:
# Create the datasets A training and validation
train_dataset = AugmentedECGDataset(valid_image_paths, labels_df, transform=train_transform)
val_dataset = AugmentedECGDataset(valid_test_images, labels_df, transform=val_transform)

In [10]:
from torch.utils.data import DataLoader

batch_size = 1
n_workers = 0

# Create DataLoaders for training and validation datasets
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=n_workers)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=n_workers)

In [18]:
for batch in train_dataloader:
    pixels = batch['pixel_values']
    labels = batch['labels']
    print(labels)
    print(pixels.min(), pixels.max(), pixels.mean(), pixels.std())
    break

tensor([[0., 0., 0., 1., 0.]])
tensor(-0.9216) tensor(0.7412) tensor(0.1186) tensor(0.4173)


/var/folders/7s/40ntfhvj5c32kvn0xr_d_wxm0000gn/T/ipykernel_2273/1043090284.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "pixel_values": torch.tensor(image, dtype=torch.float32).unsqueeze(0),


In [112]:
print("Dataset read in!")

Dataset read in!


In [113]:
class simpler_ECG_classifier(nn.Module):
    def __init__(self, num_labels=5):
        super(simpler_ECG_classifier, self).__init__()
        
        # First convolutional layer
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        # Global average pooling to reduce feature maps to 1x1
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layer for classification
        self.fc1 = nn.Linear(64, num_labels)
        
    def forward(self, x):
        # Convolutional layer with ReLU activation
        x = F.relu(self.conv1(x))
        
        # Max pooling layer
        x = self.pool(x)
        
        # Global average pooling
        x = self.global_avg_pool(x)
        
        # Flatten the tensor for the fully connected layer
        x = x.view(x.size(0), -1)
        
        # Fully connected layer for classification
        x = self.fc1(x)
        return x


In [114]:
import torch
import torch.nn as nn
from torchvision import models
import torch.nn.functional as F

class ECGClassifier(nn.Module):
    def __init__(self, num_labels=5):
        super(ECGClassifier, self).__init__()
        
        # Use ResNet18 as backbone but modify first layer for grayscale
        self.model = models.resnet18(pretrained=True)
        
        # Modify first conv layer to accept grayscale
        first_conv_weights = self.model.conv1.weight.data
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.conv1.weight.data = torch.mean(first_conv_weights, dim=1, keepdim=True)
        
        # Remove the original classifier
        num_features = self.model.fc.in_features  # This will be 512 for ResNet18
        self.model.fc = nn.Identity()
        
        # ECG-specific feature extraction - maintain 512 channels to match attention
        self.ecg_features = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),  # Changed from 256 to 512
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        
        # Multi-head classifier - adjusted for 512 input features
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),  # Changed input from 256 to 512
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_labels)
        )
        
        # Attention mechanism - keeping same dimensions
        self.attention = nn.Sequential(
            nn.Linear(512, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )
        
    def forward(self, x):
        # Extract features using backbone
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)  # [batch_size, 512, H, W]
        
        # Apply attention
        b, c, h, w = x.size()
        features = x.view(b, c, -1)  # [batch_size, 512, H*W]
        attention_weights = self.attention(features.permute(0, 2, 1))  # [batch_size, H*W, 1]
        attention_weights = F.softmax(attention_weights, dim=1)
        attended_features = torch.bmm(features, attention_weights)  # [batch_size, 512, 1]
        attended_features = attended_features.squeeze(-1)  # [batch_size, 512]
        
        # ECG-specific feature extraction
        x = self.ecg_features(x)  # Now outputs [batch_size, 512, 1, 1]
        x = x.view(x.size(0), -1)  # [batch_size, 512]
        
        # Combine attended features with ECG features
        x = x + attended_features  # Now dimensions match: both are [batch_size, 512]
        
        # Classification
        x = self.classifier(x)
        return x

def get_optimizer(model, learning_rate=0.001):
    return torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.01,
        betas=(0.9, 0.999)
    )

def get_scheduler(optimizer, num_training_steps):
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.001,
        total_steps=num_training_steps,
        pct_start=0.3
    )

In [115]:
print('Beginning model training')

Beginning model training


In [116]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
from tqdm import tqdm
import logging

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [117]:
def train_model(model, train_dataloader, val_dataloader, 
                num_epochs=10, device="cuda", patience=5,
                save_path='best_model.pth'):
    
    # Initialize optimizer and scheduler
    optimizer = get_optimizer(model)
    num_training_steps = num_epochs * len(train_dataloader)
    scheduler = get_scheduler(optimizer, num_training_steps)

    criterion = nn.BCEWithLogitsLoss()

    # Initialize early stopping
    early_stopping = EarlyStopping(patience=patience)
    
    # Track best metrics
    best_val_accuracy = 0
    best_model_state = None
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'label_precision': [],
        'label_recall': [],
        'label_f1': []
    }
    
    preds_over_time = {
        'epoch': [],
        'preds': [],
        'raw_preds': []
    }

    print(f"Training on device: {device}")
    model = model.to(device)
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        total_train_loss = 0
        train_batches = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs} [Train]')
        
        for batch in train_batches:
            # Move batch to device
            inputs = batch["pixel_values"].to(device).squeeze(1)
            labels = batch["labels"].to(device)
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            
            # Clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update weights
            optimizer.step()
            scheduler.step()
            
            # Update metrics
            total_train_loss += loss.item()
            train_batches.set_postfix({'loss': loss.item()})
        
        avg_train_loss = total_train_loss / len(train_dataloader)
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        all_preds = []
        all_labels = []
        all_raw_preds = []
        
        val_batches = tqdm(val_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs} [Val]')
        
        with torch.no_grad():
            for batch in val_batches:
                inputs = batch["pixel_values"].to(device).squeeze(1)
                labels = batch["labels"].to(device)

                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                # Get predictions
                raw_preds = torch.sigmoid(outputs).float()
                preds = (torch.sigmoid(outputs) > 0.5).float()
                
                # Update metrics
                total_val_loss += loss.item()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_raw_preds.extend(raw_preds.cpu().numpy())
        
        # Calculate validation metrics
        avg_val_loss = total_val_loss / len(val_dataloader)
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        all_raw_preds = np.array(all_raw_preds)
        
        val_accuracy = accuracy_score(all_labels.flatten(), all_preds.flatten())
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels.flatten(), 
            all_preds.flatten(), 
            average='binary'
        )
        
        # Compute label-specific metrics
        label_precision, label_recall, label_f1, _ = precision_recall_fscore_support(
            all_labels, 
            all_preds, 
            average=None
        )

        preds_over_time['epoch'].append(epoch)
        preds_over_time['preds'].append(all_preds)
        preds_over_time['raw_preds'].append(all_raw_preds)
                
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['val_precision'].append(precision)
        history['val_recall'].append(recall)
        history['val_f1'].append(f1)
        history['label_precision'].append(label_precision.tolist())
        history['label_recall'].append(label_recall.tolist())
        history['label_f1'].append(label_f1.tolist())
        
        # Save best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
            torch.save(best_model_state, save_path)
        
        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f"\nEpoch {epoch + 1}/{num_epochs} Summary:")
        print(f"Time: {epoch_time:.2f}s")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"Val Accuracy: {val_accuracy:.4f}")
        print(f"Val Precision: {precision:.4f}")
        print(f"Val Recall: {recall:.4f}")
        print(f"Val F1: {f1:.4f}")
        print(f"Label-wise Precision: {label_precision}")
        print(f"Label-wise Recall: {label_recall}")
        print(f"Label-wise F1: {label_f1}")
        
        # Early stopping check
        early_stopping(avg_val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break
    
    # Load best model
    model.load_state_dict(torch.load(save_path))
    return model, history


In [118]:
# Run the training
if __name__ == "__main__":
    # Set device
    device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
    
    # Initialize model and move to device
    model = simpler_ECG_classifier(num_labels=5)

    # Train model
    trained_model, history = train_model(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        num_epochs=5,
        device=device,
        patience=5,
        save_path='logs/best_ecg_model.pth'
    )
    
    # Save training history
    np.save('logs/training_history.npy', history)

Training on device: mps


Epoch 1/5 [Train]:   0%|          | 0/949 [00:00<?, ?it/s]/var/folders/7s/40ntfhvj5c32kvn0xr_d_wxm0000gn/T/ipykernel_1538/1043090284.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "pixel_values": torch.tensor(image, dtype=torch.float32).unsqueeze(0),
Epoch 1/5 [Val]: 100%|██████████| 50/50 [00:06<00:00,  8.31it/s]
/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: Undefine


Epoch 1/5 Summary:
Time: 130.57s
Train Loss: 0.5650
Val Loss: 0.4555
Val Accuracy: 0.8240
Val Precision: 0.0000
Val Recall: 0.0000
Val F1: 0.0000
Label-wise Precision: [0. 0. 0. 0. 0.]
Label-wise Recall: [0. 0. 0. 0. 0.]
Label-wise F1: [0. 0. 0. 0. 0.]


Epoch 2/5 [Train]:   0%|          | 0/949 [00:00<?, ?it/s]/var/folders/7s/40ntfhvj5c32kvn0xr_d_wxm0000gn/T/ipykernel_1538/1043090284.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "pixel_values": torch.tensor(image, dtype=torch.float32).unsqueeze(0),
Epoch 2/5 [Val]: 100%|██████████| 50/50 [00:06<00:00,  8.29it/s]
/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: Undefine


Epoch 2/5 Summary:
Time: 130.32s
Train Loss: 0.4602
Val Loss: 0.4604
Val Accuracy: 0.8240
Val Precision: 0.0000
Val Recall: 0.0000
Val F1: 0.0000
Label-wise Precision: [0. 0. 0. 0. 0.]
Label-wise Recall: [0. 0. 0. 0. 0.]
Label-wise F1: [0. 0. 0. 0. 0.]


...


Epoch 4/5 Summary:
Time: 130.39s
Train Loss: 0.4556
Val Loss: 0.4522
Val Accuracy: 0.8240
Val Precision: 0.0000
Val Recall: 0.0000
Val F1: 0.0000
Label-wise Precision: [0. 0. 0. 0. 0.]
Label-wise Recall: [0. 0. 0. 0. 0.]
Label-wise F1: [0. 0. 0. 0. 0.]


Epoch 5/5 [Train]:   0%|          | 0/949 [00:00<?, ?it/s]/var/folders/7s/40ntfhvj5c32kvn0xr_d_wxm0000gn/T/ipykernel_1538/1043090284.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "pixel_values": torch.tensor(image, dtype=torch.float32).unsqueeze(0),
Epoch 5/5 [Val]: 100%|██████████| 50/50 [00:06<00:00,  8.30it/s]


Epoch 5/5 Summary:
Time: 130.89s
Train Loss: 0.4555
Val Loss: 0.4524
Val Accuracy: 0.8240
Val Precision: 0.0000
Val Recall: 0.0000
Val F1: 0.0000
Label-wise Precision: [0. 0. 0. 0. 0.]
Label-wise Recall: [0. 0. 0. 0. 0.]
Label-wise F1: [0. 0. 0. 0. 0.]



/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/griffin/Documents/Data_Science_Projects/bhf_classification/bhf_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/var/folders/7s/40ntfhvj5c32kvn0xr_d_wxm0000gn/T/ipykernel_1538/2991252762.py:166: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible t

ValueError: not enough values to unpack (expected 5, got 2)

In [96]:
data = np.load('logs/training_history.npy', allow_pickle=True)
data_dict = data.item()

In [97]:
# output_content = captured_output.stdout

# with open('logs/output.log', 'w') as file: 
#     file.write(output_content)

In [100]:
preds_df = pd.DataFrame(preds, columns = ['A', 'B', 'C', 'D', 'E'])
labels_df = pd.DataFrame(labels, columns = ['A', 'B', 'C', 'D', 'E'])
raw_preds_df = pd.DataFrame(raw_preds, columns = ['A', 'B', 'C', 'D', 'E'])


In [101]:
raw_preds_df

,A,B,C,D,E
0,0.214680,0.082389,0.230502,0.231307,0.058706
1,0.258131,0.138005,0.278190,0.242497,0.096583
2,0.272081,0.134394,0.289581,0.296957,0.110524
3,0.252125,0.127500,0.271866,0.247394,0.091127
4,0.212592,0.077762,0.226788,0.240696,0.057486
5,0.231067,0.090025,0.245627,0.263333,0.070792
6,0.244039,0.101573,0.259346,0.276279,0.081752
7,0.287475,0.157071,0.306179,0.303444,0.128903
8,0.229004,0.087908,0.243878,0.262426,0.069021
9,0.277124,0.151163,0.295928,0.278842,0.116798
